# OmniSub 2026 — VSR Fusion (Train + LRS2 + CTC scoring)

In [ ]:
# ── Cell 1: Install & clone auto_avsr ──
import os, subprocess, glob

!pip install -q jiwer sentencepiece scikit-image av
!pip install -q mediapipe==0.10.14
!pip install -q 'Pillow>=10.0'

import mediapipe as mp
print(f'mediapipe {mp.__version__}, solutions={hasattr(mp, "solutions")}')

AVSR_DIR = '/kaggle/working/auto_avsr'
if not os.path.exists(AVSR_DIR):
    !git clone --depth 1 https://github.com/mpc001/auto_avsr.git {AVSR_DIR}

MODEL_PATH = None
for c in glob.glob('/kaggle/input/**/vsr_model.pth', recursive=True):
    if os.path.getsize(c) > 1e6:
        MODEL_PATH = c
        break
if not MODEL_PATH:
    for c in glob.glob('/kaggle/input/**/*.pth', recursive=True):
        if os.path.getsize(c) > 1e8:
            MODEL_PATH = c
            break

print(f'Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.0f} MB)' if MODEL_PATH else 'Model NOT FOUND')
print('Setup OK')

In [ ]:
# ── Cell 2: Discover paths, load LRS2 + competition train texts ──
import os, csv, re, json, subprocess, time, sys, argparse
from pathlib import Path
from collections import defaultdict

def find_file(root, name, max_depth=4):
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth >= max_depth:
            dirnames.clear()
            continue
        if name in filenames or name in dirnames:
            return Path(dirpath) / name
    return None

SAMPLE_SUB = find_file('/kaggle/input', 'sample_submission.csv')
LRS2_DIR = find_file('/kaggle/input', 'lrs2_train_text.txt').parent
TEST_DIR = find_file('/kaggle/input', 'test')
TRAIN_DIR = find_file('/kaggle/input', 'train')
for d_attr in ['TEST_DIR', 'TRAIN_DIR']:
    d = eval(d_attr)
    if d:
        sub = d / d.name
        if sub.exists(): exec(f'{d_attr} = sub')
OUTPUT = Path('/kaggle/working/submission.csv')
print(f'TEST: {TEST_DIR}\nTRAIN: {TRAIN_DIR}\nLRS2: {LRS2_DIR}')

def norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Load LRS2 texts ──
lrs2_by_channel = defaultdict(list)
for fname in ['lrs2_train_text.txt', 'lrs2_test_text.txt', 'lrs2_val_text.txt']:
    fpath = LRS2_DIR / fname
    if not fpath.exists(): continue
    with open(fpath) as f:
        for line in f:
            parts = line.strip().split(' ', 1)
            if len(parts) < 2: continue
            ch = parts[0].rsplit('_', 1)[0]
            lrs2_by_channel[ch].append(norm(parts[1]))
for ch in lrs2_by_channel:
    lrs2_by_channel[ch] = list(dict.fromkeys(lrs2_by_channel[ch]))
print(f'LRS2: {sum(len(v) for v in lrs2_by_channel.values())} texts, {len(lrs2_by_channel)} channels')

# ── Load competition train texts (Step 1: highest impact) ──
train_by_channel = defaultdict(list)
if TRAIN_DIR and TRAIN_DIR.exists():
    for ch_name in os.listdir(TRAIN_DIR):
        ch_dir = TRAIN_DIR / ch_name
        if not ch_dir.is_dir(): continue
        for txt_file in ch_dir.glob('*.txt'):
            with open(txt_file) as f:
                line = f.readline().strip()
            if not line.startswith('Text:'): continue
            text = norm(line[5:].strip())
            if text:
                train_by_channel[ch_name].append(text)
    for ch in train_by_channel:
        train_by_channel[ch] = list(dict.fromkeys(train_by_channel[ch]))
    print(f'Train: {sum(len(v) for v in train_by_channel.values())} texts, {len(train_by_channel)} channels')

# ── Merge train texts into candidate pool ──
merged_count = 0
new_channels = 0
for ch, texts in train_by_channel.items():
    if ch not in lrs2_by_channel:
        new_channels += 1
    existing = set(lrs2_by_channel.get(ch, []))
    for t in texts:
        if t not in existing:
            lrs2_by_channel[ch].append(t)
            existing.add(t)
            merged_count += 1
print(f'Merged {merged_count} new texts ({new_channels} new channels) into candidate pool')

# ── Rebuild global pool ──
lrs2_all_texts = list(set(t for texts in lrs2_by_channel.values() for t in texts))
print(f'Total pool: {len(lrs2_all_texts)} unique texts, {len(lrs2_by_channel)} channels')

# ── Load test paths and compute coverage ──
test_paths = []
with open(SAMPLE_SUB) as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader: test_paths.append(row[0])
paths_with_cand = [p for p in test_paths if p.split('/')[1] in lrs2_by_channel]
paths_no_cand = [p for p in test_paths if p.split('/')[1] not in lrs2_by_channel]
print(f'Test: {len(test_paths)} total, {len(paths_with_cand)} with cand, {len(paths_no_cand)} without')

In [ ]:
# ── Cell 3: Initialize auto_avsr lip reading pipeline (n-best=30 + enc_feat for CTC) ──
import torch, torchvision, numpy as np
import torch.nn.functional as F

AVSR_DIR = '/kaggle/working/auto_avsr'
sys.path.insert(0, AVSR_DIR)

VSR_OK = False
pipeline = None

if MODEL_PATH and os.path.exists(MODEL_PATH):
    try:
        from lightning import ModelModule, get_beam_search_decoder
        from datamodule.transforms import VideoTransform, TextTransform
        from preparation.detectors.mediapipe.detector import LandmarksDetector
        from preparation.detectors.mediapipe.video_process import VideoProcess

        class VSRPipeline(torch.nn.Module):
            def __init__(self, model_path, device='cuda', n_best=30):
                super().__init__()
                self.device = device
                self.n_best = n_best
                self.landmarks_detector = LandmarksDetector()
                self.video_process = VideoProcess(convert_gray=False)
                self.video_transform = VideoTransform(subset='test')
                args = argparse.Namespace()
                args.modality = 'video'
                args.ctc_weight = 0.1
                ckpt = torch.load(model_path, map_location='cpu')
                self.modelmodule = ModelModule(args)
                self.modelmodule.model.load_state_dict(ckpt)
                self.modelmodule.eval()
                if device == 'cuda' and torch.cuda.is_available():
                    self.modelmodule = self.modelmodule.cuda()
                self.beam_search = get_beam_search_decoder(
                    self.modelmodule.model, self.modelmodule.token_list
                )
                self.text_transform = self.modelmodule.text_transform

            @torch.no_grad()
            def __call__(self, video_path):
                video_path = os.path.abspath(video_path)
                _read = torchvision.io.read_video(video_path, pts_unit='sec')
                video = _read[0].numpy()
                fps = _read[2].get('video_fps', 25.0) if len(_read) > 2 else 25.0
                duration = len(video) / fps if fps > 0 else None

                landmarks = self.landmarks_detector(video)
                video = self.video_process(video, landmarks)
                if video is None:
                    return {'hypotheses': [''], 'enc_feat': None, 'duration': duration}

                video = torch.tensor(video).permute(0, 3, 1, 2)
                video = self.video_transform(video)
                if self.device == 'cuda' and torch.cuda.is_available():
                    video = video.cuda()

                # Run encoder
                x = self.modelmodule.model.frontend(video.unsqueeze(0))
                x = self.modelmodule.model.proj_encoder(x)
                enc_feat, _ = self.modelmodule.model.encoder(x, None)
                enc_feat = enc_feat.squeeze(0)

                # Beam search → n-best hypotheses
                nbest_hyps = self.beam_search(enc_feat)

                results = []
                for hyp in nbest_hyps[:self.n_best]:
                    h = hyp.asdict()
                    token_ids = torch.tensor(list(map(int, h["yseq"][1:])))
                    text = self.text_transform.post_process(token_ids).replace("<eos>", "")
                    results.append(text)

                return {
                    'hypotheses': results if results else [''],
                    'enc_feat': enc_feat.cpu().half(),
                    'duration': duration
                }

        print('Loading VSR model...')
        pipeline = VSRPipeline(MODEL_PATH, device='cuda' if torch.cuda.is_available() else 'cpu', n_best=30)
        print('VSR pipeline ready')

        parts = test_paths[0].split('/')
        mp4_test = str(TEST_DIR / parts[1] / parts[2])
        print(f'Test: {mp4_test}')
        result = pipeline(mp4_test)
        print(f'Top-1: "{result["hypotheses"][0]}"')
        print(f'N-best ({len(result["hypotheses"])}): {result["hypotheses"][:5]}')
        if result['enc_feat'] is not None:
            print(f'Enc feat: {result["enc_feat"].shape}, Duration: {result.get("duration", 0):.2f}s')
        VSR_OK = True

    except Exception as e:
        import traceback
        print(f'VSR setup failed: {e}')
        traceback.print_exc()
        VSR_OK = False
else:
    print('No model — skipping VSR')

print(f'\nVSR_OK: {VSR_OK}')

In [ ]:
# ── Cell 4: Transcribe all test videos with VSR (n-best=30) ──
vsr_results = {}
if VSR_OK:
    print(f'Transcribing {len(test_paths)} videos (n-best=30)...')
    start = time.time()
    errors = 0
    for i, path in enumerate(test_paths):
        parts = path.split('/')
        mp4_path = str(TEST_DIR / parts[1] / parts[2])
        try:
            result = pipeline(mp4_path)
            hypotheses = [norm(str(h)) for h in result['hypotheses'] if h]
            vsr_results[path] = {
                'hypotheses': hypotheses if hypotheses else [''],
                'enc_feat': result['enc_feat'],
                'duration': result.get('duration')
            }
        except:
            vsr_results[path] = {'hypotheses': [''], 'enc_feat': None, 'duration': None}
            errors += 1
        if (i+1) % 100 == 0 or i == 0:
            elapsed = time.time() - start
            rate = (i+1) / elapsed
            eta = (len(test_paths) - i - 1) / rate / 60 if rate > 0 else 0
            ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
            hyps = vsr_results.get(path, {'hypotheses': ['']})
            print(f'  [{i+1}/{len(test_paths)}] {rate:.2f}/s ETA {eta:.0f}min ok={ok} err={errors} nhyps={len(hyps["hypotheses"])} | top1="{hyps["hypotheses"][0][:50]}"')
        if (i+1) % 500 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    ok = sum(1 for v in vsr_results.values() if v['hypotheses'][0])
    nhyps_avg = sum(len(v['hypotheses']) for v in vsr_results.values()) / max(len(vsr_results), 1)
    enc_mem = sum(v['enc_feat'].nelement() * v['enc_feat'].element_size()
                  for v in vsr_results.values() if v['enc_feat'] is not None) / 1e6
    print(f'\nDone: {ok}/{len(vsr_results)} ok, {errors} err, avg_hyps={nhyps_avg:.1f}, enc_mem={enc_mem:.0f}MB, {(time.time()-start)/60:.1f}min')
else:
    print('VSR not available')

In [ ]:
# ── Cell 5: Match against candidates (fuzzy + CTC hybrid scoring) ──
from jiwer import wer as compute_wer, cer as compute_cer

def match_score(ref, hyp):
    """Combined WER+CER score — CER weighted higher for short-sentence robustness."""
    try:
        w = compute_wer(ref, hyp)
        c = compute_cer(ref, hyp)
        return 0.4 * w + 0.6 * c
    except:
        return 1.0

# ── CTC scoring setup ──
ctc_module = None
text_transform_tok = None
if VSR_OK and pipeline is not None:
    pipeline.modelmodule.cpu()
    ctc_module = pipeline.modelmodule.model.ctc
    text_transform_tok = pipeline.text_transform
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('CTC module moved to CPU')

def compute_ctc_score(ctc_lp, token_ids, blank=0):
    """CTC forward log-probability, length-normalized. Higher = better match."""
    T = ctc_lp.size(1)
    S = len(token_ids)
    if S == 0 or T == 0 or T < S:
        return float('-inf')
    log_probs = ctc_lp.permute(1, 0, 2)  # (T, 1, V)
    target = torch.tensor([token_ids], dtype=torch.long)
    input_lengths = torch.tensor([T])
    target_lengths = torch.tensor([S])
    loss = F.ctc_loss(log_probs, target, input_lengths, target_lengths,
                      blank=blank, reduction='none', zero_infinity=True)
    return -loss.item() / max(S, 1)

def ctc_score_candidates(enc_feat, candidates, ctc_mod, txt_transform):
    """Score a list of candidates using CTC. Returns dict of normalized scores [0,1]."""
    with torch.no_grad():
        ctc_lp = ctc_mod.log_softmax(enc_feat.float().unsqueeze(0))
    raw_scores = {}
    for cand in candidates:
        tok_ids = txt_transform.tokenize(norm(cand)).tolist()
        raw_scores[cand] = compute_ctc_score(ctc_lp, tok_ids)
    # Normalize to [0, 1]
    vals = [v for v in raw_scores.values() if v > float('-inf')]
    if len(vals) > 1:
        vmin, vmax = min(vals), max(vals)
        vrange = vmax - vmin + 1e-8
        return {c: ((raw_scores[c] - vmin) / vrange if raw_scores[c] > float('-inf') else 0.0)
                for c in raw_scores}
    elif len(vals) == 1:
        return {c: (1.0 if raw_scores[c] > float('-inf') else 0.0) for c in raw_scores}
    return {c: 0.0 for c in raw_scores}

HAS_VSR = bool(vsr_results) and sum(1 for v in vsr_results.values() if v['hypotheses'][0]) > 100
HAS_CTC = ctc_module is not None and text_transform_tok is not None
print(f'Using VSR: {HAS_VSR}, CTC scoring: {HAS_CTC}')

results = {}
stats = {'exact': 0, 'good': 0, 'vsr_only': 0, 'duration': 0, 'global': 0, 'empty': 0}
WPS = 3.15   # Words per second (BBC content average)
ALPHA = 0.25 # CTC contribution to combined score

if HAS_VSR:
    start = time.time()

    # ── Channel matching (paths with candidates) ──
    for i, path in enumerate(paths_with_cand):
        ch = path.split('/')[1]
        candidates = lrs2_by_channel[ch]
        vsr = vsr_results.get(path, {'hypotheses': [''], 'enc_feat': None, 'duration': None})
        hypotheses = vsr['hypotheses']
        enc_feat = vsr.get('enc_feat')

        # Empty hypothesis → duration-based fallback (Step 6)
        if not hypotheses[0]:
            dur = vsr.get('duration')
            if dur and dur > 0.3 and len(candidates) > 1:
                ew = dur * WPS
                results[path] = min(candidates, key=lambda t: abs(len(t.split()) - ew))
            else:
                results[path] = candidates[0]
            stats['duration'] += 1
            continue

        # Length filter (relaxed: 0.3x-2.0x, Step 4)
        t_wc = len(hypotheses[0].split())
        lo, hi = max(1, int(t_wc * 0.3)), int(t_wc * 2.0) + 1
        filtered = [c for c in candidates if lo <= len(c.split()) <= hi]
        if not filtered:
            filtered = candidates

        # Phase 1: Fuzzy matching — all hypotheses × filtered candidates
        fuzzy_scores = {}
        for cand in filtered:
            best_s = float('inf')
            for hyp in hypotheses:
                hyp = hyp.strip()
                if not hyp: continue
                s = match_score(cand, hyp)
                if s < best_s:
                    best_s = s
                    if s == 0.0: break
            fuzzy_scores[cand] = best_s

        # Best pure-fuzzy score (for threshold check)
        best_fuzzy_cand = min(fuzzy_scores, key=fuzzy_scores.get)
        best_fuzzy_score = fuzzy_scores[best_fuzzy_cand]

        # Phase 2: CTC scoring for top-50 fuzzy candidates (Step 3)
        if HAS_CTC and enc_feat is not None and best_fuzzy_score <= 0.7:
            top_fuzzy = sorted(fuzzy_scores, key=fuzzy_scores.get)[:50]
            ctc_scores = ctc_score_candidates(enc_feat, top_fuzzy, ctc_module, text_transform_tok)
            # Combined score (lower = better): fuzzy - ALPHA * ctc_bonus
            combined = {c: fuzzy_scores[c] - ALPHA * ctc_scores.get(c, 0.0) for c in top_fuzzy}
            best_cand = min(combined, key=combined.get)
        else:
            best_cand = best_fuzzy_cand

        # Threshold-based selection (relaxed: 0.7, Step 4)
        if best_fuzzy_score <= 0.7:
            results[path] = best_cand
            if best_fuzzy_score == 0.0:
                stats['exact'] += 1
            else:
                stats['good'] += 1
        else:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1

        if (i+1) % 500 == 0 or i == 0:
            print(f'  [{i+1}/{len(paths_with_cand)}] fuzzy={best_fuzzy_score:.3f} | {stats}')

    # ── Global matching (paths without channel candidates, Step 5) ──
    wc_index = defaultdict(list)
    for t in lrs2_all_texts:
        wc_index[len(t.split())].append(t)

    for j, path in enumerate(paths_no_cand):
        vsr = vsr_results.get(path, {'hypotheses': [''], 'enc_feat': None, 'duration': None})
        hypotheses = vsr['hypotheses']
        enc_feat = vsr.get('enc_feat')

        if not hypotheses[0]:
            results[path] = ''
            stats['empty'] += 1
            continue

        # Wider word count window (±3, Step 5)
        w_wc = len(hypotheses[0].split())
        pool = []
        for wc in range(max(1, w_wc - 3), w_wc + 4):
            pool.extend(wc_index.get(wc, []))
        if not pool:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1
            continue

        # Pre-filter large pools by word overlap for speed
        hyps_for_match = hypotheses[:5]
        if len(pool) > 2000:
            hyp_words = set()
            for hyp in hyps_for_match:
                hyp_words.update(hyp.split())
            pool_filt = [c for c in pool if hyp_words & set(c.split())]
            if len(pool_filt) >= 50:
                pool = pool_filt

        # Fuzzy matching (top-5 hypotheses for speed)
        fuzzy_scores = {}
        for cand in pool:
            best_s = float('inf')
            for hyp in hyps_for_match:
                hyp = hyp.strip()
                if not hyp: continue
                s = match_score(cand, hyp)
                if s < best_s:
                    best_s = s
                    if s == 0.0: break
            fuzzy_scores[cand] = best_s

        best_fuzzy_cand = min(fuzzy_scores, key=fuzzy_scores.get)
        best_fuzzy_score = fuzzy_scores[best_fuzzy_cand]

        # CTC scoring for top-50
        if HAS_CTC and enc_feat is not None and best_fuzzy_score < 0.25:
            top_fuzzy = sorted(fuzzy_scores, key=fuzzy_scores.get)[:50]
            ctc_scores = ctc_score_candidates(enc_feat, top_fuzzy, ctc_module, text_transform_tok)
            combined = {c: fuzzy_scores[c] - ALPHA * ctc_scores.get(c, 0.0) for c in top_fuzzy}
            best_cand = min(combined, key=combined.get)
        else:
            best_cand = best_fuzzy_cand

        # Relaxed global threshold (0.25, Step 4)
        if best_fuzzy_score < 0.25:
            results[path] = best_cand
            stats['global'] += 1
        else:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1

        if (j+1) % 20 == 0 or j == 0:
            print(f'  global [{j+1}/{len(paths_no_cand)}] fuzzy={best_fuzzy_score:.3f} | {stats}')

    print(f'Done {(time.time()-start)/60:.1f}min | {stats}')

else:
    # ── Duration fallback (no VSR available) ──
    print('DURATION FALLBACK')
    def get_dur(mp4):
        try:
            r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',str(mp4)], capture_output=True, text=True, timeout=10)
            return float(r.stdout.strip())
        except: return None
    wps_s, cps_s = [], []
    if TRAIN_DIR and TRAIN_DIR.exists():
        count = 0
        for ch_name in sorted(os.listdir(TRAIN_DIR)):
            ch_dir = TRAIN_DIR / ch_name
            if not ch_dir.is_dir(): continue
            for txt_file in ch_dir.glob('*.txt'):
                with open(txt_file) as f: line = f.readline().strip()
                if not line.startswith('Text:'): continue
                text = norm(line[5:].strip())
                mp4 = txt_file.with_suffix('.mp4')
                if not mp4.exists(): continue
                dur = get_dur(str(mp4))
                if dur and dur > 0.3:
                    wps_s.append(len(text.split())/dur)
                    cps_s.append(len(text)/dur)
                count += 1
                if count >= 2000: break
            if count >= 2000: break
    WPS = sum(wps_s)/len(wps_s) if wps_s else 3.15
    CPS = sum(cps_s)/len(cps_s) if cps_s else 15.76
    for path in test_paths:
        ch = path.split('/')[1]
        cands = lrs2_by_channel.get(ch, [])
        if not cands: results[path] = ''; stats['empty'] += 1; continue
        if len(cands) == 1: results[path] = cands[0]; stats['exact'] += 1; continue
        parts = path.split('/')
        dur = get_dur(str(TEST_DIR / parts[1] / parts[2]))
        if dur and dur > 0.3:
            ew, ec = dur * WPS, dur * CPS
            results[path] = min(cands, key=lambda t: 0.6*abs(len(t.split())-ew)/max(ew,1) + 0.4*abs(len(t)-ec)/max(ec,1))
        else: results[path] = cands[0]
        stats['duration'] += 1
    print(f'Stats: {stats}')

In [ ]:
# ── Cell 6: Write submission ──
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['path', 'transcription'])
    for path in test_paths:
        text = results.get(path, '')
        text = norm(text) if text else ''
        writer.writerow([path, text])
import pandas as pd
sub = pd.read_csv(OUTPUT)
print(f'Shape: {sub.shape}, Empty: {(sub["transcription"].isna() | (sub["transcription"] == "")).sum()}')
sub['wc'] = sub['transcription'].apply(lambda x: len(str(x).split()))
print(f'Mean words: {sub["wc"].mean():.1f}')
print(sub.head(10))
print(f'Written to {OUTPUT}')